# Ablation — HIT-ASR's design switches and the number of experts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/ablation.ipynb)

What each design switch of HIT-ASR does, and how the router scales with the number of experts. Three sources, each
named in its tables:

| § | Source | What it shows |
|---|---|---|
| **1** | the **hold-out exploration** of HIT-ASR on each trio: configurations drawn at random from a space in which the fusion, the sharing of Stage 1, the objective and the input normalisation are all hyperparameters, each trained on the hold-out's fit rows and scored on its validation rows | per level of each switch: how often it collapses onto the best single expert, how much of the gap between the best single expert and the oracle its typical and its best configurations close, its size and its fitting time |
| **2** | the **folds of Table 3** | HIT-ASR against the pooled router, which reads the same encoder states averaged over time, and the hard-CE control |
| **3** | the **hold-out**, with K = 3, 5 and 10 experts | HIT-ASR and MLP-pool with their selected configurations on larger pools |

Sections 1 and 3 are computed on the hold-out, on which the main table never tests.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the records on the Hub | a CPU, seconds |
| **2** | the study of section 3 retrained (sections 1 and 2 are read in either case) | a GPU, ~50 min |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
from hitasr.ablation import Ablation
from hitasr.configs import ABLATION
from labkit.pretty import show

LEVEL = 1          # 1 or 2
DEVICE = "cuda"    # level 2 trains on it

ablation = Ablation(ABLATION, level=LEVEL, device=DEVICE)

ami_sdm: Cohere-Transcribe, Granite-Speech-4.1-2b, Parakeet-TDT-0.6b-v3 — 141 exploration trials; main configuration fusion=concat, share_stage1=False, loss_preset=soft_ce_only, input_norm=none
earnings22: Cohere-Transcribe, Kyutai-STT-2.6b, Parakeet-TDT-0.6b-v3 — 141 exploration trials; main configuration fusion=mean, share_stage1=False, loss_preset=wer+hard, input_norm=none
peoples_speech: Granite-Speech-4.1-2b, Parakeet-CTC-1.1b, Parakeet-TDT-0.6b-v3 — 155 exploration trials; main configuration fusion=mean, share_stage1=True, loss_preset=soft_ce_only, input_norm=standardize


## 1. Design switches

Per corpus and level of each switch, over every configuration drawn: `collapsed` is the share that closed at most 1 %
of the gap between the best single expert and the oracle; `gap_median`, `gap_q75` and `gap_best` summarise the share
of that gap closed on the hold-out's validation rows; `top_share` is the median share of clips routed to the busiest
expert; `params_M` and `fit_s` are medians. ◀ marks the level the main search selected.

In [3]:
show(*ablation.switches())

ami_sdm — HIT-ASR's switches on the hold-out
  [fusion]
    level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
   bridge 14      0.500       0.054    0.194     0.216      0.895     3.289 92.587     
   concat  6      0.500       0.015    0.082     0.100      0.930     2.870 85.544    ◀
     mean 18      0.222       0.083    0.179     0.283      0.620     2.756 76.653     
self_attn 22      0.364       0.046    0.117     0.245      0.840     1.833 96.429     
  [share_stage1]
level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M   fit_s main
False 27      0.259       0.070    0.117     0.257       0.83     3.295 100.419    ◀
 True 33      0.455       0.052    0.140     0.283       0.85     1.713  82.483     
  [loss_preset]
        level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M   fit_s main
          all  7      0.429       0.109    0.138     0.156      0.850     3.282 245.198     
      default  6      0.333       0.112    0.215     0.245      0.535     2.459 132.141     
 hard_ce_only  9      0.556       0.000    0.052     0.053      0.950     1.962 103.020     
 soft_ce_only 10      0.200       0.068    0.108     0.141      0.525     1.807  76.196    ◀
     wer+hard  8      0.000       0.090    0.100     0.140      0.910     2.233  67.660     
wer+soft_tau1  8      0.125       0.192    0.207     0.214      0.545     1.633  72.816     
     wer_only 12      0.750      -0.210   -0.102     0.283      1.000     3.453  68.507     
  [input_norm]
      level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M   fit_s main
       none 27      0.370       0.062    0.170     0.245      0.800     2.093  97.273    ◀
      scale 17      0.471       0.047    0.101     0.213      0.850     2.504  74.113     
standardize 16      0.250       0.065    0.144     0.283      0.835     2.818 101.555

earnings22 — HIT-ASR's switches on the hold-out
  [fusion]
    level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
   bridge 14      0.571       0.000    0.287     0.339      0.910     3.190 32.413     
   concat  6      0.167       0.268    0.317     0.406      0.805     2.706 27.946     
     mean 18      0.278       0.239    0.318     0.419      0.800     2.593 19.005    ◀
self_attn 22      0.364       0.273    0.334     0.412      0.710     1.767 29.402     
  [share_stage1]
level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
False 27      0.370       0.276    0.308     0.419       0.71     3.216 29.531    ◀
 True 33      0.364       0.219    0.318     0.412       0.79     1.582 22.851     
  [loss_preset]
        level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
          all  7      0.143       0.312    0.330     0.349      0.790     3.216 67.187     
      default  6      0.500       0.024    0.265     0.348      0.695     2.393 30.615     
 hard_ce_only  9      0.222       0.219    0.339     0.373      0.810     1.897 27.080     
 soft_ce_only 10      0.000       0.298    0.340     0.419      0.490     1.709 27.750     
     wer+hard  8      0.125       0.309    0.345     0.412      0.695     2.135 18.191    ◀
wer+soft_tau1  8      0.375       0.051    0.193     0.303      0.685     1.568 13.502     
     wer_only 12      1.000       0.000    0.000     0.003      1.000     3.257 28.125     
  [input_norm]
      level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
       none 27      0.444       0.048    0.311     0.412      0.830     2.028 40.732    ◀
      scale 17      0.294       0.288    0.325     0.419      0.780     2.373 27.080     
standardize 16      0.312       0.271    0.302     0.349      0.745     2.687 19.762

peoples_speech — HIT-ASR's switches on the hold-out
  [fusion]
    level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
   bridge 14      0.357       0.144    0.215     0.441      0.850     3.239 49.726     
   concat  6      0.167       0.274    0.399     0.434      0.800     2.788 27.731     
     mean 18      0.167       0.294    0.426     0.595      0.660     2.674 30.786    ◀
self_attn 22      0.273       0.338    0.429     0.532      0.795     1.800 37.515     
  [share_stage1]
level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
False 27      0.259       0.346    0.441     0.595       0.75     3.249 38.480     
 True 33      0.242       0.229    0.385     0.555       0.82     1.647 31.778    ◀
  [loss_preset]
        level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
          all  7      0.143       0.258    0.380     0.431      0.840     3.249 68.305     
      default  6      0.333       0.417    0.441     0.517      0.530     2.426 51.154     
 hard_ce_only  9      0.111       0.218    0.288     0.366      0.820     1.930 31.401     
 soft_ce_only 10      0.000       0.419    0.507     0.595      0.425     1.758 41.931    ◀
     wer+hard  8      0.125       0.415    0.458     0.517      0.795     2.184 17.245     
wer+soft_tau1  8      0.000       0.230    0.311     0.536      0.620     1.600 26.146     
     wer_only 12      0.833       0.000    0.000     0.255      1.000     3.355 36.759     
  [input_norm]
      level  n  collapsed  gap_median  gap_q75  gap_best  top_share  params_M  fit_s main
       none 27      0.259       0.209    0.386     0.536       0.83     2.061 34.984     
      scale 17      0.294       0.277    0.424     0.555       0.78     2.439 34.047     
standardize 16      0.188       0.356    0.433     0.595       0.78     2.752 32.061    ◀

### Seed noise

How much one configuration's result moves when it is retrained with another seed; a difference between levels
smaller than this is not a difference.

In [4]:
show(*ablation.noise())

index,n_configs,gap_sd_median,status_flips
ami_sdm,43,0.042,0.256
earnings22,43,0.035,0.070
peoples_speech,50,0.064,0.060


fusion,42,43,44
bridge,0.207,0.156,0.131
concat,0.091,0.023,-0.030
mean,0.106,0.170,0.081
self_attn,0.082,0.064,0.069


fusion,42,43,44
bridge,0.295,0.299,0.325
concat,0.316,0.285,0.295
mean,0.303,0.274,0.330
self_attn,0.310,0.272,0.295


fusion,42,43,44
bridge,0.207,0.380,0.344
concat,0.366,0.426,0.345
mean,0.385,0.404,0.402
self_attn,0.417,0.400,0.439


### Across the corpora

In [5]:
ablation.across()

Every switch level, averaged over the corpora
  [fusion]
    level  collapsed  gap_median  gap_best  params_M  chosen_by_main
   bridge      0.476       0.066     0.332     3.239               0
   concat      0.278       0.186     0.313     2.788               1
     mean      0.222       0.205     0.432     2.674               2
self_attn      0.333       0.219     0.396     1.800               0
  [share_stage1]
level  collapsed  gap_median  gap_best  params_M  chosen_by_main
False      0.296       0.231     0.423     3.254               2
 True      0.354       0.167     0.416     1.647               1
  [loss_preset]
        level  collapsed  gap_median  gap_best  params_M  chosen_by_main
          all      0.238       0.226     0.312     3.249               0
      default      0.389       0.184     0.370     2.426               0
 hard_ce_only      0.296       0.146     0.264     1.930               0
 soft_ce_only      0.067       0.262     0.385     1.758               2
     wer+hard      0.083       0.271     0.356     2.184               1
wer+soft_tau1      0.167       0.158     0.351     1.600               0
     wer_only      0.861      -0.070     0.180     3.355               0
  [input_norm]
      level  collapsed  gap_median  gap_best  params_M  chosen_by_main
       none      0.358       0.107     0.398     2.061               2
      scale      0.353       0.204     0.395     2.439               0
standardize      0.250       0.230     0.409     2.752               1

## 2. On the folds of Table 3

Rows of each corpus's Table 3 (means over its ten folds), with the p-value of each row against HIT-ASR from its
Table 6.

In [6]:
show(*ablation.main_rows())

arm,label,corpus_wer,corpus_wer_sd,gap_closed,acc,top_share,p_vs_hit_asr
bsm,Best single expert,0.1842,0.0022,0.0000,0.7358,1.0000,0.0076
mlp_pool_hard_ce,MLP-pool (hard CE),0.1780,0.0027,0.0920,0.7655,0.8747,0.0309
adastt_ce,ADASTT,0.1823,0.0022,0.0286,0.7493,0.9615,0.0100
mlp_pool,MLP-pool,0.1744,0.0021,0.1450,0.7839,0.7479,0.0342
hit_asr,HIT-ASR,0.1719,0.0030,0.1818,0.7973,0.5409,—
oracle,Oracle,0.1168,0.0013,1.0000,1.0000,0.7358,0.0000


arm,label,corpus_wer,corpus_wer_sd,gap_closed,acc,top_share,p_vs_hit_asr
bsm,Best single expert,0.1279,0.0024,0.0000,0.6801,1.0000,0.0033
mlp_pool_hard_ce,MLP-pool (hard CE),0.1235,0.0036,0.1160,0.7175,0.7327,0.0182
adastt_ce,ADASTT,0.1282,0.0033,-0.0062,0.6889,0.8981,0.0161
mlp_pool,MLP-pool,0.1214,0.0036,0.1735,0.7313,0.6670,0.0404
hit_asr,HIT-ASR,0.1198,0.0027,0.2135,0.7325,0.7345,—
oracle,Oracle,0.0900,0.0018,1.0000,1.0000,0.6411,0.0001


arm,label,corpus_wer,corpus_wer_sd,gap_closed,acc,top_share,p_vs_hit_asr
bsm,Best single expert,0.1739,0.0026,0.0000,0.6733,1.0000,0.0013
mlp_pool_hard_ce,MLP-pool (hard CE),0.1650,0.0026,0.1908,0.6983,0.8325,0.0029
adastt_ce,ADASTT,0.1678,0.0023,0.1305,0.6916,0.9168,0.0083
mlp_pool,MLP-pool,0.1629,0.0032,0.2363,0.7050,0.6985,0.0327
hit_asr,HIT-ASR,0.1580,0.0019,0.3401,0.7156,0.4548,—
oracle,Oracle,0.1274,0.0013,1.0000,1.0000,0.6733,0.0000


## 3. Number of experts

The trio is grown one expert at a time, each time by the expert that lowers the oracle WER most on the hold-out's fit
rows, so the larger pools contain the smaller ones. HIT-ASR and MLP-pool keep their selected configurations; each is
trained on the hold-out's fit rows and scored on its validation rows, once per model seed.

In [7]:
show(*ablation.pool_size())

peoples_speech — K experts on the hold-out, mean over seeds (42, 43, 44)
pools: K=3: Granite-Speech-4.1-2b, Parakeet-CTC-1.1b, Parakeet-TDT-0.6b-v3; K=5: Granite-Speech-4.1-2b, Parakeet-CTC-1.1b, Parakeet-TDT-0.6b-v3, Whisper-large-v3, Whisper-large-v3-turbo; K=10: Granite-Speech-4.1-2b, Parakeet-CTC-1.1b, Parakeet-TDT-0.6b-v3, Whisper-large-v3, Whisper-large-v3-turbo, Qwen3-ASR-1.7B, Cohere-Transcribe, Kyutai-STT-2.6b, Distil-Whisper-large-v3.5, Voxtral-Mini-3B
  [3]
     arm  gap_closed  gap_sd  corpus_wer    acc  top_share  bsm_wer  oracle_wer  params_M    fit_s
 HIT-ASR       0.581  0.0263      0.1592 0.7117     0.4741   0.1997        0.13    2.0287 153.4153
MLP-pool       0.521  0.0210      0.1634 0.7260     0.7061   0.1997        0.13   44.0832   7.4060
  [5]
     arm  gap_closed  gap_sd  corpus_wer    acc  top_share  bsm_wer  oracle_wer  params_M    fit_s
 HIT-ASR      0.0024  0.0424      0.1601 0.6033     0.2644   0.1602      0.1143    3.0148 291.6260
MLP-pool     -0.0651  0.0803      0.1632 0.6129     0.6167   0.1602      0.1143   59.8241   6.3062
  [10]
     arm  gap_closed  gap_sd  corpus_wer    acc  top_share  bsm_wer  oracle_wer  params_M    fit_s
 HIT-ASR      0.0978  0.0451      0.1559 0.5480     0.1740   0.1613      0.1065    5.0869 562.2468
MLP-pool      0.0303  0.0451      0.1596 0.5461     0.6266   0.1613      0.1065   92.8850  13.8555

## Record

In [8]:
ablation.save();

wrote ablation.json
